# TotalEnergies and the price of oil

## Main question

To what extent do TotalEnergies' results depend on the Brent price, and how much do they fall when oil prices drop?

## Sub-questions

1. **Do the results follow the Brent price?**
   How did adjusted net income, sales and production evolve between 2011 and 2025, compared with the average Brent price? How strongly are these indicators correlated with Brent?
2. **How much is a higher oil price worth?**
   On average, how much does adjusted net income change when Brent changes by 10 USD/bbl? How does this estimate compare with the official sensitivity published by TotalEnergies?
3. **What does the company lose when oil prices fall?**
   During the three downturns of the period (2014-2016, 2020 and after 2022), how far did results fall from their peak? Did they fall faster or slower than the Brent price?
4. **Does the downstream business cushion falling oil prices?**
   How do upstream and downstream (refining, chemicals, marketing) results react to changes in Brent? Does downstream offset part of the upstream losses when prices fall?
5. **Why do net income and adjusted net income diverge in some years?**
   Which years show the largest gaps, and what explains them?

## Data

All data come from TotalEnergies' annual Factbooks (2014 to 2025 editions) and, for a few figures, from its quarterly press releases and SEC filings. Column definitions, sources and checks are detailed in the README.

| File | Content |
| --- | --- |
| `totalenergies_annual_2011_2025.csv` | One row per year: Brent, sales, net income, adjusted net income, production |
| `segments_annual_2011_2025.csv` | Adjusted net operating income by year and by segment |
| `totalenergies_quarterly_2011_2025.csv` | One row per quarter: Brent, liquids price, net income, adjusted net income |
| `segments_quarterly_2011_2025.csv` | Adjusted net operating income by quarter and by segment |

Amounts are in millions of US dollars, Brent in dollars per barrel.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 20)

In [2]:
annual = pd.read_csv("../data/totalenergies_annual_2011_2025.csv")
segments = pd.read_csv("../data/segments_annual_2011_2025.csv")
quarterly = pd.read_csv("../data/totalenergies_quarterly_2011_2025.csv")
segments_q = pd.read_csv("../data/segments_quarterly_2011_2025.csv")

for name, df in [("annual", annual), ("segments", segments), ("quarterly", quarterly), ("segments_q", segments_q)]:
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

annual: 15 rows, 6 columns
segments: 61 rows, 6 columns
quarterly: 60 rows, 6 columns
segments_q: 236 rows, 6 columns


To give all charts the same style, I use a small formatting function. It adds the title, the axis labels, a light grid and the source.

In [3]:
sns.set_theme(style="white")
plt.rcParams["figure.figsize"] = (10, 4.5)

BLUE, ORANGE, GREEN, GREY = "#4C72B0", "#DD8452", "#55A868", "#8C8C8C"


def format_axes(ax: plt.Axes, title: str, xlabel: str, ylabel: str,
                source: str | None = "Source: TotalEnergies Factbooks") -> None:
    """Apply the same style to every chart in the notebook.

    Set source to None for all but one chart of a figure, so the source appears only once.
    """
    ax.set_title(title, loc="left", fontsize=13, fontweight="bold")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    if source:
        ax.figure.text(0.01, -0.03, source, fontsize=8, color="gray")

## 1. Exploring and preparing the data

### 1.1 Structure of the files

In [4]:
display(annual.head())
display(segments_q.head())

,year,brent_avg_usd_per_bbl,sales_musd,net_income_musd,adjusted_net_income_musd,production_kboe_per_day
0,2011,111.3,257084,17400,15948,2346
1,2012,111.7,257037,13648,15772,2300
2,2013,108.7,251725,11228,14292,2299
3,2014,99.0,236122,4244,12837,2146
4,2015,52.4,165357,5087,10518,2347


,quarter,quarter_end_date,segment,group,adjusted_net_operating_income_musd,source
0,2011Q1,2011-03-31,Upstream,Upstream & Gas,NaN,not available
1,2011Q1,2011-03-31,Refining & Chemicals,Downstream,NaN,not available
2,2011Q1,2011-03-31,Marketing & Services,Downstream,NaN,not available
3,2011Q2,2011-06-30,Upstream,Upstream & Gas,NaN,not available
4,2011Q2,2011-06-30,Refining & Chemicals,Downstream,NaN,not available


In [5]:
for name, df in [("annual", annual), ("segments", segments), ("quarterly", quarterly), ("segments_q", segments_q)]:
    print(f"--- {name} ---")
    print(df.dtypes, "\n")

--- annual ---
year                          int64
brent_avg_usd_per_bbl       float64
sales_musd                    int64
net_income_musd               int64
adjusted_net_income_musd      int64
production_kboe_per_day       int64
dtype: object 

--- segments ---
year                                  int64
segment                                 str
group                                   str
adjusted_net_operating_income_musd    int64
factbook_edition                      int64
sheet                                   str
dtype: object 

--- quarterly ---
quarter                              str
quarter_end_date                     str
brent_avg_usd_per_bbl            float64
avg_liquids_price_usd_per_bbl    float64
net_income_musd                  float64
adjusted_net_income_musd         float64
dtype: object 

--- segments_q ---
quarter                                   str
quarter_end_date                          str
segment                                   str
group              

In [6]:
print(segments["group"].value_counts(), "\n")
print(segments["segment"].value_counts())

group
Upstream & Gas    31
Downstream        30
Name: count, dtype: int64 

segment
Refining & Chemicals                  15
Marketing & Services                  15
Exploration & Production              13
Integrated LNG                         5
Integrated Power                       5
Integrated Gas, Renewables & Power     4
Upstream                               2
Gas, Renewables & Power                2
Name: count, dtype: int64


The segment names change over time because TotalEnergies reorganized its business segments several times. The `group` column (Upstream & Gas / Downstream) is the one to use to compare years.

### 1.2 Dates

In the quarterly files, `quarter_end_date` is read as text. I convert it to a date and extract the year, which I will need to group quarters by year.

In [7]:
for df in [quarterly, segments_q]:
    df["quarter_end_date"] = pd.to_datetime(df["quarter_end_date"])
    df["year"] = df["quarter_end_date"].dt.year

quarterly.dtypes

quarter                                     str
quarter_end_date                 datetime64[us]
brent_avg_usd_per_bbl                   float64
avg_liquids_price_usd_per_bbl           float64
net_income_musd                         float64
adjusted_net_income_musd                float64
year                                      int32
dtype: object

### 1.3 Missing values

In [8]:
for name, df in [("annual", annual), ("segments", segments), ("quarterly", quarterly), ("segments_q", segments_q)]:
    print(f"{name}: {df.isna().sum().sum()} missing values")

annual: 0 missing values
segments: 0 missing values
quarterly: 0 missing values
segments_q: 24 missing values


In [9]:
missing = segments_q[segments_q["adjusted_net_operating_income_musd"].isna()]
missing.groupby(["year", "segment"]).size().unstack()

segment,Marketing & Services,Refining & Chemicals,Upstream
year,,,
2011,4,4,4
2012,4,4,4


Only `segments_q` has missing values: the results by segment for every quarter of 2011 and 2012 (24 rows).

**Decision: I keep them.** They only concern the quarterly split by segment in 2011 and 2012, which is outside the three downturns studied, and the annual file covers these years by segment. Filling them would mean inventing data, and pandas already ignores missing values in calculations. The one exception is the quarterly pivot table in section 5.3, where I drop them first: otherwise, `sum` would turn the empty quarters into zeros.

### 1.4 More readable units

Results are in millions of US dollars: USD 36,197 million reads better as USD 36.2 billion. I add columns in billions for the charts.

In [10]:
for df in [annual, quarterly]:
    df["net_income_busd"] = df["net_income_musd"] / 1000
    df["adjusted_net_income_busd"] = df["adjusted_net_income_musd"] / 1000
annual["sales_busd"] = annual["sales_musd"] / 1000

for df in [segments, segments_q]:
    df["adjusted_noi_busd"] = df["adjusted_net_operating_income_musd"] / 1000

annual.head()

,year,brent_avg_usd_per_bbl,sales_musd,net_income_musd,adjusted_net_income_musd,production_kboe_per_day,net_income_busd,adjusted_net_income_busd,sales_busd
0,2011,111.3,257084,17400,15948,2346,17.400,15.948,257.084
1,2012,111.7,257037,13648,15772,2300,13.648,15.772,257.037
2,2013,108.7,251725,11228,14292,2299,11.228,14.292,251.725
3,2014,99.0,236122,4244,12837,2146,4.244,12.837,236.122
4,2015,52.4,165357,5087,10518,2347,5.087,10.518,165.357
